# T2 — Semantic ICD linking (bge-m3 + reranker) trên Colab

Build embedding index cho ICD-10-CM rồi đo hit@k trên dev gold.

**Trước khi chạy:** Runtime → Change runtime type → **T4 GPU** → Save.

Chạy lần lượt các cell. Xong gửi lại khối `=== ICD-10 (CHẨN_ĐOÁN) ===` cho P2.

In [1]:
# 1) Kiểm tra GPU (phải thấy Tesla T4). Nếu lỗi -> chưa bật GPU ở Runtime.
!nvidia-smi -L

GPU 0: Tesla T4 (UUID: GPU-d3c462d6-f8e3-d2cd-598e-f713e50656d2)


In [ ]:
# 2) Clone repo (nhánh Duckun). Repo private -> dán GitHub token (PAT).
#    Tạo PAT: GitHub > Settings > Developer settings > Personal access tokens (quyền đọc repo).
PAT = ""  #@param {type:"string"}
BRANCH = "Duckun"  #@param {type:"string"}
REPO = "tienndat1904/viettel-medreason"  #@param {type:"string"}
import os
if os.path.isdir('viettel-medreason'):
    %cd viettel-medreason
    !git pull
else:
    !git clone -b {BRANCH} https://{PAT}@github.com/{REPO}.git
    %cd viettel-medreason
!git log --oneline -1

In [2]:
# 3) Cài thư viện (Colab đã có torch). Nếu lỗi version, đổi sang: !pip -q install -U FlagEmbedding
!pip -q install FlagEmbedding==1.3.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.8/177.8 kB 7.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 542.0/542.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 128.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 947.5/947.5 kB 61.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 172.0/172.0 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 46.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 122.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.19.0 requires huggingface-hub<2.0,>=1.2.0, but you have h

In [3]:
# 4) (Tuỳ chọn) Nạp lại index đã lưu ở Drive để KHỎI build lại. Bỏ qua nếu build lần đầu.
from google.colab import drive
drive.mount('/content/drive')
import os, shutil
src = '/content/drive/MyDrive/viettel_kb'
for f in ('icd10cm_emb.npy', 'icd10cm_index_meta.parquet'):
    p = os.path.join(src, f)
    if os.path.exists(p):
        shutil.copy(p, 'data/kb/'); print('đã nạp', f)
    else:
        print('chưa có', f, '-> sẽ build ở cell 5')

Mounted at /content/drive
chưa có icd10cm_emb.npy -> sẽ build ở cell 5
chưa có icd10cm_index_meta.parquet -> sẽ build ở cell 5


In [4]:
# 5) Build index (lần đầu tải bge-m3 ~2.3GB, ~vài phút trên T4).
#    Nếu đã nạp từ Drive ở cell 4 thì có thể bỏ qua cell này.
#    Hết VRAM -> thêm: --batch-size 64
!python src/kb/build_icd_index.py

python3: can't open file '/content/src/kb/build_icd_index.py': [Errno 2] No such file or directory


In [ ]:
# 6) Bật semantic + đo trên 30 file dev gold (embed query + rerank cần GPU).
!sed -i 's/backend: lexical/backend: semantic/' configs/config.yaml
!python src/eval/eval_linking.py

In [ ]:
# 7) (Nên) Lưu index vào Drive để phiên sau khỏi build lại (dùng ở cell 4).
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/viettel_kb
!cp data/kb/icd10cm_emb.npy data/kb/icd10cm_index_meta.parquet /content/drive/MyDrive/viettel_kb/
print('Đã lưu index vào Drive/MyDrive/viettel_kb')

## Gửi lại cho P2
Copy khối kết quả ở cell 6:
```
=== ICD-10 (CHẨN_ĐOÁN) ===
  hit@k ... = 0.???
  top1  ... = 0.???
```
So với bản lexical hiện tại **0.495** để quyết định tinh chỉnh `icd_top_k_retrieve` / `icd_top_k_return` / `min_confidence`.